In [1]:
import os
import sys
import warnings
from functools import partial

from gymnasium import Env

In [2]:
import torch
from tensordict import TensorDict
from tensordict.nn import TensorDictModule
from torchrl.envs import GymWrapper
from torchrl.modules import QValueActor, DuelingCnnDQNet, EGreedyModule
from torchrl.data.replay_buffers import LazyMemmapStorage, PrioritizedReplayBuffer
from torchrl.record import CSVLogger

In [83]:
from abc import ABC, abstractmethod
from typing import Callable, Optional, Any

In [82]:
import gymnasium as gym
import ale_py
from gymnasium.wrappers import AtariPreprocessing, FrameStackObservation
from stable_baselines3.common.atari_wrappers import FireResetEnv, EpisodicLifeEnv, ClipRewardEnv

In [5]:
warnings.filterwarnings("ignore")
gym.register_envs(ale_py)
if os.path.abspath("package") not in sys.path: sys.path.append(os.path.abspath("package"))

In [6]:
from package.environment import GymPreprocessing, create_breakout_env, create_breakout_env_gym
from package.dqn_types import ModelParameters, PathsParameters, EnvSpaceName
from package.utils import fill_buffer, init_collector
from package.video import VideoPlayer, Recorder, unstack_frames
from package.Logger import SmartLogger, LogsConfig, Logger
from package.modules import (Model,
                             Scale,
                             Optimizer,
                             Trainer,
                             initialize_weights,
                             init_lazy_layers,
                             n_parameters)

In [7]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv
from stable_baselines3.common.callbacks import BaseCallback

In [8]:
from stable_baselines3.common.evaluation import evaluate_policy

In [92]:
class Evaluator(ABC):
    @abstractmethod
    def __call__(self) -> dict[str, int | float | bool]:
        pass

In [93]:
class EvaluateReward(Evaluator):
    def __init__(self, network: Model, env: Env, freq: int = 1, n_episodes: int = 10, deterministic: bool = False):
        self.model: Model = network
        self.env: Env = env
        self.freq: int = freq
        self.n_episodes: int = n_episodes
        self.deterministic: bool = deterministic

    def __call__(self) -> dict[str, int | float]:
        mean_reward, std_reward = evaluate_policy(self.model, self.env, self.n_episodes, self.deterministic)
        return dict(mean_reward=mean_reward, std_reward=std_reward)

In [105]:
class Callback(BaseCallback):
    def __init__(self, *evaluators: Evaluator, stop_criterion: Optional[Evaluator] = None, verbose: int = 0):
        super().__init__(verbose)
        self.evaluators: tuple[Evaluator] = evaluators
        self.stop_criterion: Optional[Evaluator] = stop_criterion
        self.initial_state: bool = True

    def _on_step(self) -> bool:
        if self.initial_state:
            self.initial_state = False
            return True
        for evaluator in self.evaluators:
            if self.n_calls % evaluator.freq: continue
            for key, value in evaluator().items():
                self.logger.record(key, value)
        if self.stop_criterion is not None:
            criterion: dict[str, int | float] = self.stop_criterion()
            if all(criterion.values()):
                return False
        return True

    def _on_rollout_start(self) -> None:
        print("--- Метрики после обновления модели ---")
        # self.model.logger.record("metrick", 10.)
        print(self.logger.name_to_value)

In [106]:
env_prep = GymPreprocessing(
    partial(AtariPreprocessing,
            noop_max=20,
            frame_skip=4,
            terminal_on_life_loss=False,
            screen_size=64,
            grayscale_newaxis=False),
    partial(EpisodicLifeEnv),
    partial(FireResetEnv),
    partial(ClipRewardEnv),
    partial(FrameStackObservation, stack_size=4)
)

In [107]:
build_env: Callable[[], Env] = lambda: create_breakout_env_gym(transform=env_prep)
envir: DummyVecEnv = DummyVecEnv(list(build_env for _ in range(8)))

In [108]:
model = PPO(
    "CnnPolicy",
    envir,
    verbose=0,
    learning_rate=1e-4,
    n_steps=1,
    batch_size=32,
    n_epochs=10,
    device="mps",
    # tensorboard_log="./tensorboard_log"
)

In [109]:
model

In [110]:
_ = model.learn(total_timesteps=8 * 1 * 10, callback=Callback())

--- Метрики после обновления модели ---
defaultdict(<class 'float'>, {})
1
--- Метрики после обновления модели ---
defaultdict(<class 'float'>, {'train/learning_rate': 0.0001, 'train/entropy_loss': np.float64(-1.3849725484848023), 'train/policy_gradient_loss': np.float64(-0.01861310452222824), 'train/value_loss': np.float64(0.0056859125055780165), 'train/approx_kl': np.float32(0.004081771), 'train/clip_fraction': np.float64(0.0), 'train/loss': -0.0383077934384346, 'train/explained_variance': -0.39272987842559814, 'train/n_updates': 10, 'train/clip_range': 0.2})
2
--- Метрики после обновления модели ---
defaultdict(<class 'float'>, {'train/learning_rate': 0.0001, 'train/entropy_loss': np.float64(-1.3701200008392334), 'train/policy_gradient_loss': np.float64(-0.01897435300052166), 'train/value_loss': np.float64(0.0005764905845353496), 'train/approx_kl': np.float32(0.015006088), 'train/clip_fraction': np.float64(0.0), 'train/loss': -0.04189528897404671, 'train/explained_variance': 0.04134

In [70]:
model.logger.name_to_value

defaultdict(float,
            {'train/learning_rate': 0.0001,
             'train/entropy_loss': np.float64(-1.386293888092041),
             'train/policy_gradient_loss': np.float64(1.4901161193847656e-08),
             'train/value_loss': np.float64(1.4681296306662261e-05),
             'train/approx_kl': np.float32(0.0),
             'train/clip_fraction': np.float64(0.0),
             'train/loss': 7.355549314524978e-06,
             'train/explained_variance': 0.966733992099762,
             'train/n_updates': 1,
             'train/clip_range': 0.2})

In [19]:
def make_env(seed=0):
    def _init():
        env = create_breakout_env_gym(transform=env_prep)
        return env

    return _init

In [35]:
make_atari_env("BreakoutNoFrameskip-v4", n_envs=8, seed=0).reset().shape

(8, 84, 84, 1)

In [42]:
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack


def train():
    # 1. Создаем окружение с базовыми обертками Atari (NoopReset, MaxAndSkip и т.д.)
    # n_envs=8 позволяет обучать агента в 8 окнах одновременно (сильно ускоряет процесс)
    env = make_atari_env("BreakoutNoFrameskip-v4", n_envs=8, seed=0)
    print(env.reset().shape)

    # 2. Склеиваем 4 кадра для понимания динамики движения
    env = VecFrameStack(env, n_stack=4)
    print(env.reset().shape)

    # 3. Создаем модель
    # CnnPolicy — специальная нейросеть для обработки изображений
    model = PPO(
        "CnnPolicy",
        env,
        verbose=1,
        learning_rate=1e-4,
        batch_size=256,
        n_steps=128,  # количество шагов до обновления градиента
        device="cuda"  # используй "cpu", если нет видеокарты NVIDIA
    )

    # 4. Запускаем обучение (начни с 1 000 000 шагов для видимого результата)
    print("Начинаю обучение...")
    model.learn(total_timesteps=10)

    # 5. Сохраняем модель
    model.save("ppo_breakout_model")
    print("Модель сохранена.")

In [43]:
train()

(8, 84, 84, 1)
(8, 84, 84, 4)
Using cpu device
Wrapping the env in a VecTransposeImage.
Начинаю обучение...
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 519      |
|    ep_rew_mean     | 0        |
| time/              |          |
|    fps             | 1552     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 1024     |
---------------------------------
Модель сохранена.


In [ ]:


def train():
    # 1. Создаем функцию для генерации одиночной среды
    def make_env(seed=0):
        def _init():
            env = create_breakout_env_gym(transform=env_prep)
            return env

        return _init

    # 2. Создаем параллельные среды (например, 8 штук)
    n_envs = 8
    env = SubprocVecEnv([make_env(seed=i) for i in range(n_envs)])
    # Установка seed для каждой среды через метод seed (или в reset в более новых версиях)
    env.seed(0)

    # 3. Склеиваем 4 кадра для понимания динамики движения
    # Примечание: env_prep уже содержит FrameStackObservation, 
    # поэтому VecFrameStack может быть избыточен, если в env_prep уже стоит stack_size=4.
    # Но для демонстрации совместимости оставляем.
    # env = VecFrameStack(env, n_stack=4) 

    # 4. Создаем модель
    model = PPO(
        "CnnPolicy",
        env,
        verbose=1,
        learning_rate=1e-4,
        batch_size=256,
        n_steps=128,
        device="cuda"
    )

    # 5. Запускаем обучение
    print("Начинаю обучение...")
    model.learn(total_timesteps=1_000_000)

    # 6. Сохраняем модель
    model.save("ppo_breakout_model")
    print("Модель сохранена.")


if __name__ == "__main__":
    train()